# DTP Managed Roads — Bronze Ingestion & Data Quality Assessment

## Purpose
Ingest the DTP Managed Roads GeoJSON dataset and assess the quality of the
raw source before downstream transformation.

## Bronze Layer Objectives
- Preserve the source data without business transformation
- Validate dataset structure
- Assess completeness
- Assess identifier uniqueness
- Validate geospatial features
- Identify potential data-quality issues
- Establish whether the dataset is suitable for Silver processing

## Source
Dataset: DTP Managed Roads
Format: GeoJSON
Domain: Transport Asset / Road Network

In [0]:
%pip install geopandas pyogrio


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import geopandas as gpd
import pandas as pd

geojson_path = "/Volumes/dtp_data/dtp_schema/dtp_geojson/_published_roads_dtp_managed_roads_dtp_managed_roads.geojson"

roads_bronze = gpd.read_file(geojson_path)

print("=== DTP MANAGED ROADS — SOURCE SUMMARY ===")
print(f"Object type : {type(roads_bronze).__name__}")
print(f"Records     : {len(roads_bronze):,}")
print(f"Columns     : {len(roads_bronze.columns)}")
print(f"CRS         : {roads_bronze.crs}")

print("\nGeometry types:")
print(roads_bronze.geometry.geom_type.value_counts())

=== DTP MANAGED ROADS — SOURCE SUMMARY ===
Object type : GeoDataFrame
Records     : 90,797
Columns     : 24
CRS         : EPSG:4326

Geometry types:
LineString    90797
Name: count, dtype: int64


# Data Profiling

In [0]:
profile = pd.DataFrame({
    "column": roads_bronze.columns,
    "dtype": roads_bronze.dtypes.astype(str).values,
    "missing_count": roads_bronze.isna().sum().values,
    "missing_pct": (
        roads_bronze.isna().mean() * 100
    ).round(2).values,
    "unique_values": [
        roads_bronze[col].nunique(dropna=True)
        for col in roads_bronze.columns
    ]
})

display(profile)

column,dtype,missing_count,missing_pct,unique_values
OBJECTID,int32,0,0.0,90797
DEC_NAME,object,0,0.0,886
DEC_TYPE,object,0,0.0,25
RD_NAME,object,0,0.0,886
RD_TYPE,object,0,0.0,25
LOCAL_NAME,object,0,0.0,2687
LOCAL_TYPE,object,0,0.0,28
ALT_NAME,object,0,0.0,915
ALT_TYPE,object,0,0.0,19
ALT2_NAME,object,0,0.0,525


### Data Profiling Finding

The initial profiling identified no conventional NULL values in the source
attributes.

However, a zero NULL count does not necessarily indicate complete data.
Text fields may contain empty or whitespace-only strings that are not
recognised as NULL by Pandas.

A separate blank-string assessment is therefore required before determining
attribute completeness.

## Blank String Assessment

Assess text attributes for empty or whitespace-only values that are not
captured by conventional NULL checks.

In [0]:
# Assess blank / whitespace-only values in text columns

blank_results = []

for col in roads_bronze.select_dtypes(include="object").columns:

    blank_count = (
        roads_bronze[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    blank_results.append({
        "column": col,
        "blank_count": blank_count,
        "blank_pct": round(
            blank_count / len(roads_bronze) * 100,
            2
        )
    })

blank_profile = pd.DataFrame(blank_results)

blank_profile = (
    blank_profile[
        blank_profile["blank_count"] > 0
    ]
    .sort_values(
        "blank_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(blank_profile)

column,blank_count,blank_pct
ALT2_SUFFIX,90661,99.85
ALT_SUFFIX,90531,99.71
RD_SUFFIX,89870,98.98
LOCAL_SUFFIX,88264,97.21
ALT2_NAME,80656,88.83
ALT2_TYPE,80650,88.82
ALT_TYPE,68367,75.3
ALT_NAME,68333,75.26
DEC_TYPE,1321,1.45
RD_TYPE,1321,1.45


## Identifier & Duplicate Assessment

Validate whether OBJECTID uniquely identifies source road features and
determine whether duplicate records are present.

No source records are modified during this assessment.

In [0]:
print("=== DUPLICATE / UNIQUENESS CHECK ===")

print("Total records:",
      len(roads_bronze))

print("Unique OBJECTIDs:",
      roads_bronze["OBJECTID"].nunique())

print("Duplicate OBJECTIDs:",
      roads_bronze["OBJECTID"].duplicated().sum())

print("Fully duplicated rows:",
      roads_bronze.duplicated().sum())

=== DUPLICATE / UNIQUENESS CHECK ===
Total records: 90797
Unique OBJECTIDs: 90797
Duplicate OBJECTIDs: 0
Fully duplicated rows: 0


### Finding

- 90,797 source records were assessed.
- OBJECTID contains 90,797 unique values.
- No duplicate OBJECTIDs were identified.
- No fully duplicated records were identified.

**Assessment:** OBJECTID is unique within the current source extract.
No duplicate remediation is required before Silver processing.

## Geospatial Quality Assessment

Validate the spatial integrity of the road network before downstream
transformation and analysis.

The assessment checks geometry type, missing and empty geometries,
geometry validity, and coordinate reference system (CRS).

In [0]:
print("=== GEOSPATIAL QUALITY CHECK ===")

print("Geometry types:")
print(roads_bronze.geometry.geom_type.value_counts())

print("\nMissing geometries:",
      roads_bronze.geometry.isna().sum())

print("Empty geometries:",
      roads_bronze.geometry.is_empty.sum())

print("Invalid geometries:",
      (~roads_bronze.geometry.is_valid).sum())

print("\nCRS:",
      roads_bronze.crs)

=== GEOSPATIAL QUALITY CHECK ===
Geometry types:
LineString    90797
Name: count, dtype: int64

Missing geometries: 0
Empty geometries: 0
Invalid geometries: 0

CRS: EPSG:4326


### Finding

- All 90,797 features are represented as LineString geometries.
- No missing geometries were identified.
- No empty geometries were identified.
- No invalid geometries were identified.
- The source CRS is EPSG:4326.

**Assessment:** The source geometry passes the initial structural
quality checks and is suitable for downstream Silver processing.

The source CRS will be retained in Bronze. Any reprojection required
for distance or road-length calculations will be performed downstream.

In [0]:
display(roads_bronze.head(5))

,OBJECTID,DEC_NAME,DEC_TYPE,RD_NAME,RD_TYPE,LOCAL_NAME,LOCAL_TYPE,ALT_NAME,ALT_TYPE,ALT2_NAME,ALT2_TYPE,RD_SUFFIX,LOCAL_SUFFIX,ALT_SUFFIX,ALT2_SUFFIX,RD_NUM,RD_SECTION,CLASSN,PROFILE,SRNS,RMANUM,RMACLASS,LOCALITY,geometry
0,1,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,,,,,,,,,5248,01,MR,1,N,5248,AO,WEST MELBOURNE,"LINESTRING (144.90811 -37.80662, 144.90826 -37..."
1,2,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,,,,,,,,,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,"LINESTRING (144.90912 -37.80679, 144.90909 -37..."
2,3,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,,,,,,,,,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,"LINESTRING (144.90967 -37.80691, 144.90966 -37..."
3,4,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,,,,,,,,,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,"LINESTRING (144.90909 -37.80696, 144.90912 -37..."
4,5,MACKENZIE,ROAD,MACKENZIE,ROAD,SIMS,STREET,,,,,,,,,5248,01,MR,8,N,5248,AO,WEST MELBOURNE,"LINESTRING (144.90909 -37.80696, 144.909 -37.8..."


## Persist Bronze Dataset

Persist the validated DTP Managed Roads source dataset for downstream Silver
transformation.

The Bronze layer retains the published source values without business
transformation or imputation.

In [0]:
bronze_path = "/Volumes/dtp_data/dtp_schema/dtp_geojson/managed_roads_bronze.geojson"

roads_bronze.to_file(
    bronze_path,
    driver="GeoJSON"
)

print("Bronze saved to:")
print(bronze_path)

Bronze saved to:
/Volumes/dtp_data/dtp_schema/dtp_geojson/managed_roads_bronze.geojson


In [0]:
bronze_check = gpd.read_file(bronze_path)

print("=== BRONZE OUTPUT VERIFICATION ===")

print(f"Records : {len(bronze_check):,}")
print(f"Columns : {len(bronze_check.columns)}")
print(f"CRS     : {bronze_check.crs}")

# Validate persisted Bronze against the ingested source
assert len(bronze_check) == len(roads_bronze), \
    "Record count changed during Bronze persistence"

assert len(bronze_check.columns) == len(roads_bronze.columns), \
    "Column count changed during Bronze persistence"

assert bronze_check["OBJECTID"].duplicated().sum() == 0, \
    "Duplicate OBJECTIDs detected after persistence"

assert bronze_check.geometry.isna().sum() == 0, \
    "Missing geometries detected after persistence"

print("\nBronze persistence validation: PASS")

=== BRONZE OUTPUT VERIFICATION ===
Records : 90,797
Columns : 24
CRS     : EPSG:4326

Bronze persistence validation: PASS
